# Progress Tab

> Progress bar, stage message, elapsed time, and job status badge.

In [ ]:
#| default_exp components.tabs.progress_tab

In [ ]:
#| export
import time
from typing import Optional

from fasthtml.common import Div, Span, Progress, Script, FT

from cjm_fasthtml_daisyui.components.feedback.progress import progress as progress_cls, progress_colors
from cjm_fasthtml_daisyui.components.data_display.badge import badge, badge_colors
from cjm_fasthtml_daisyui.utilities.semantic_colors import text_dui, bg_dui
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, h
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight, font_family
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import flex_display, items, justify, gap
from cjm_fasthtml_tailwind.core.base import combine_classes

from cjm_fasthtml_job_monitor.html_ids import JobMonitorHtmlIds

## Status Badge

In [ ]:
#| export
_STATUS_BADGE_COLORS = {
    'pending': badge_colors.info,
    'running': badge_colors.primary,
    'completed': badge_colors.success,
    'failed': badge_colors.error,
    'cancelled': badge_colors.warning,
}

def _render_status_badge(
    status: str,  # Job status string
) -> FT:  # Badge element
    """Render a colored badge for job status."""
    color = _STATUS_BADGE_COLORS.get(status, badge_colors.info)
    return Span(status.upper(), cls=combine_classes(badge, color, font_size.xs))

## Elapsed Time

In [ ]:
#| export
def _format_elapsed(
    started_at: Optional[float],   # Unix timestamp when job started
    completed_at: Optional[float], # Unix timestamp when job completed
) -> str:  # Formatted elapsed time string
    """Format elapsed time as M:SS."""
    if started_at is None:
        return "--:--"
    end = completed_at if completed_at else time.time()
    elapsed = max(0, int(end - started_at))
    mins = elapsed // 60
    secs = elapsed % 60
    return f"{mins}:{secs:02d}"

def _elapsed_timer_script(
    ids: JobMonitorHtmlIds,      # Element IDs
    started_at: Optional[float], # Unix timestamp
) -> FT:  # Script element for client-side timer
    """Generate JS for client-side elapsed time updates."""
    if started_at is None:
        return Span()
    return Script(f"""
    (function() {{
        const startedAt = {started_at};
        const el = document.getElementById('{ids.elapsed}');
        if (!el) return;
        const iv = setInterval(() => {{
            if (!document.getElementById('{ids.elapsed}')) {{ clearInterval(iv); return; }}
            const elapsed = Math.floor(Date.now()/1000 - startedAt);
            const mins = Math.floor(elapsed / 60);
            const secs = elapsed % 60;
            el.textContent = mins + ':' + secs.toString().padStart(2, '0');
        }}, 1000);
    }})();
    """)

## render_progress_tab

In [ ]:
#| export
def render_progress_tab(
    ids: JobMonitorHtmlIds,                # Element IDs
    status: str = 'pending',               # Job status
    progress_value: float = 0.0,           # 0.0 to 1.0
    status_message: str = '',              # Stage message
    started_at: Optional[float] = None,    # Unix timestamp
    completed_at: Optional[float] = None,  # Unix timestamp
) -> FT:  # Progress tab content
    """Render progress tab content."""
    pct = max(0, min(100, int(progress_value * 100)))

    return Div(
        # Status + Elapsed row
        Div(
            _render_status_badge(status),
            Span(
                _format_elapsed(started_at, completed_at),
                id=ids.elapsed,
                cls=combine_classes(font_family.mono, font_size.sm, text_dui.base_content),
            ),
            cls=combine_classes(flex_display, items.center, justify.between, m.b(3)),
        ),
        # Progress bar
        Progress(
            value=str(pct),
            max="100",
            id=ids.progress_bar,
            cls=combine_classes(progress_cls, progress_colors.primary, w.full, h(2)),
        ),
        # Stage message
        Div(
            Span(status_message or 'Waiting...', cls=combine_classes(font_size.sm, text_dui.base_content)),
            Span(f"{pct}%", cls=combine_classes(font_size.sm, font_weight.semibold, font_family.mono)),
            cls=combine_classes(flex_display, items.center, justify.between, m.t(2)),
        ),
        # Client-side timer (only when running)
        _elapsed_timer_script(ids, started_at) if status == 'running' else Span(),
        cls=p(4),
    )

In [ ]:
# Test render_progress_tab
from cjm_fasthtml_job_monitor.html_ids import JobMonitorHtmlIds

ids = JobMonitorHtmlIds(prefix="jm")

# Pending state
result = render_progress_tab(ids, status='pending', progress_value=0.0)
assert 'PENDING' in str(result)
assert '0%' in str(result)
print("Progress tab (pending): OK")

# Running state
result = render_progress_tab(
    ids, status='running', progress_value=0.45,
    status_message='Running forced alignment...', started_at=time.time() - 30,
)
assert 'RUNNING' in str(result)
assert '45%' in str(result)
assert 'Running forced alignment...' in str(result)
print("Progress tab (running): OK")

# Completed state
result = render_progress_tab(
    ids, status='completed', progress_value=1.0,
    status_message='Alignment complete.', started_at=time.time() - 60, completed_at=time.time(),
)
assert 'COMPLETED' in str(result)
assert '100%' in str(result)
print("Progress tab (completed): OK")

Progress tab (pending): OK
Progress tab (running): OK
Progress tab (completed): OK


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()